# Imports & Parameters

In [2]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'


## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

data_dumper_path = '/Users/E2074/local-datasets/customer/appography/appography_data_dump/'
local_database = '/Users/E2074/local-datasets/customer/appography/appography_v2/bangalore/database/'

/Users/E2074/analytics/customer/Appography/appography_demographic_dataset


In [4]:
##Parameter 
start_date = '20240601'
end_date = '20240630'
city = 'Bangalore'

city_service = f"""
    select 
        service_level city_service
    from 
        datasets.service_mapping 
    where 
        city_display_name = '{city}' 
        and service_type != 'delivery'  
        and service_category != 'delivery'
"""
df_city_service = pd.read_sql(city_service, connection)
service = ', '.join("'" + value + "'" for value in sorted(df_city_service['city_service']))

In [5]:
print("\033[1m Date Range : \033[0m", start_date, " to ", end_date)
print("\033[1m City : \033[0m", city)
print("\033[1m Service : \033[0m", service)

 Date Range :  20240601  to  20240630
 City :  Bangalore
 Service :  'Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite'


# Datasets

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

/Users/E2074/analytics/customer/Appography/appography_demographic_dataset


## Customer base

### Metric data

In [6]:
print('Query 1 running..')
start_time = time.time()

active_customer_base = f"""

    WITH fe_tbl AS (
    
        SELECT 
            yyyymmdd,
            platform,
            service_detail_id,
            user_id customer_id,
            fare_estimate_id,
            quarter_hour fe_quarter_hour
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CabPremium', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        ),
        
        rr_net_tbl as (
        
        SELECT   
            yyyymmdd,
            service_detail_id,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            quarter_hour rr_quarter_hour
        FROM 
            orders.order_logs_snapshot ols
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city_name = '{city}'
            AND service_obj_service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CabPremium', 'CityAuto', 'Link', 'Lite')
        ),
        
        merged AS (
        
        SELECT
            fe_tbl.*,
            rr_net_tbl.order_id,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.rr_quarter_hour
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.yyyymmdd = rr_net_tbl.yyyymmdd
            AND fe_tbl.customer_id = rr_net_tbl.customer_id
            AND fe_tbl.service_detail_id = rr_net_tbl.service_detail_id
        )

        SELECT
            platform,
            customer_id,
            COUNT(DISTINCT fare_estimate_id) AS fe_count,
            COUNT(DISTINCT order_id) AS rr_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN fare_estimate_id END) morning_fe,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN fare_estimate_id END) evening_fe,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN order_id END) morning_rr,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN order_id END) evening_rr

        FROM 
            merged
        GROUP BY 1,2
        
"""
df_active_customer_base = pd.read_sql(active_customer_base, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_active_customer_base.head(1)

Query 1 running..



KeyboardInterrupt



In [ ]:
# df_active_customer_base.to_csv(local_database + 'raw_active_customer_base_june.csv', index=False)
df_active_customer_base = pd.read_csv(local_database + 'raw_active_customer_base_june.csv')

### Segment data

In [ ]:
print('Query 2 running..')
start_time = time.time()
customer_segments = f"""

     SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_lifetime_rides life_time_rides,
        taxi_lifetime_stage life_time_stage,
        taxi_service_affinity service_affinity

    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date =  DATE_FORMAT(DATE_PARSE('{end_date}', '%Y%m%d'), '%Y-%m-%d') -- + INTERVAL '1' DAY 
        AND taxi_recency_segment != 'INACTIVE' 
        
"""
df_customer_segments = pd.read_sql(customer_segments, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_segments.head(1)

In [ ]:
# df_customer_segments.to_csv(local_database + 'raw_customer_segments_june.csv', index=False)
df_customer_segments = pd.read_csv(local_database + 'raw_customer_segments_june.csv')

### Age and gender data

In [ ]:
print('Query 3 running..')
start_time = time.time()

customer_demo_data = f"""

        WITH fe_customer AS (

        SELECT 
            user_id customer_id
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CabPremium', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        GROUP BY 1
        ),

        demographic_data AS ( 

        SELECT
            customerId customer_id,
            CASE WHEN age BETWEEN 15 AND 60 THEN age ELSE 0 END age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            datasets.customer_single_view
        )

        SELECT
            a.customer_id,
            COALESCE(age, 0) age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            fe_customer a
        LEFT JOIN 
            demographic_data b
            on a.customer_id = b.customer_id        
"""

df_customer_demo_data = pd.read_sql(customer_demo_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_demo_data.head(1)

In [ ]:
# df_customer_demo_data.to_csv(local_database + 'raw_customer_demo_data_june.csv', index=False)
df_customer_demo_data = pd.read_csv(local_database + 'raw_customer_demo_data_june.csv')

### Use-case data

In [ ]:
print('Query 4 running..')
start_time = time.time()

customer_use_case_data = f"""

    WITH use_case AS (

        SELECT
            hex_10, 
            usecase_tag
        FROM
            (
            SELECT 
                hex_10, 
                combined_final_usecase_accuracy as usecase_tag,
                ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
            FROM
                hive.experiments_internal.combined_usecase_hex_10_level
            WHERE 
                aoi = 'Bangalore, India'
            )
        WHERE   
            seq_no = 1
        ),

        active_customers AS (

        SELECT 
            customer_id,
            order_id,
            drop_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'first_half_drop' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON drop_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 8 
            AND cast(substr(quarter_hour,1,2) as real) <= 12    
            AND city_name = '{city}'
            
        GROUP BY 1,2,3,4

        UNION 

            SELECT 
            customer_id,
            order_id,
            pickup_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'second_half_pickup' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON pickup_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 13
            AND cast(substr(quarter_hour,1,2) as real) <= 21
            AND city_name = '{city}'
        GROUP BY 1,2,3,4

        ),

        merge AS (

        SELECT
            customer_id,
            time_period,
            usecase_tag,
            orders,
            ROW_NUMBER() OVER(PARTITION BY customer_id,time_period ORDER BY orders DESC) seq_no
        FROM 
            (
            SELECT
                customer_id,
                time_period,
                usecase_tag,
                COUNT(DISTINCT order_id) orders
            FROM 
                active_customers
            GROUP BY 1,2,3
            )
        -- WHERE 
        --     LOWER(usecase_tag) != 'unknown'
        )

        SELECT
            customer_id,
            COALESCE(first_half_drop_primary, 'NA') first_half_drop_primary,
            COALESCE(first_half_drop_secondary, 'NA') first_half_drop_secondary,
            COALESCE(second_half_pickup_primary, 'NA') second_half_pickup_primary,
            COALESCE(second_half_pickup_secondary, 'NA') second_half_pickup_secondary
        FROM
        (
        SELECT
            customer_id,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 1 THEN usecase_tag END) first_half_drop_primary,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 2 THEN usecase_tag END) first_half_drop_secondary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 1 THEN usecase_tag END) second_half_pickup_primary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 2 THEN usecase_tag END) second_half_pickup_secondary
        FROM 
            merge 
        WHERE 
            seq_no in (1,2)
        GROUP BY 1
        ) 
"""

df_customer_use_case_data = pd.read_sql(customer_use_case_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_use_case_data.head(1)

In [ ]:
# df_customer_use_case_data.to_csv(local_database + 'raw_customer_use_case_data_june.csv', index=False)
df_customer_use_case_data = pd.read_csv(local_database + 'raw_customer_use_case_data_june.csv')

### Appography data

In [ ]:
local_database

In [ ]:
print('Query 5 running..')
start_time = time.time()

"""
date_range = pd.date_range(start=start_date, end=end_date, freq='D').strftime('%Y%m%d')
file_names = [f"users_app_list_{date}.csv" for date in date_range]
csv_files = [file for file in os.listdir(data_dumper_path) if file.endswith('.csv') and file in file_names]
#csv_files = ['users_app_list_20240325.csv'] # testing...

dfs = []

#Loop through each CSV file, read it into a DataFrame, and append to the list
for file in csv_files:
    file_path = os.path.join(data_dumper_path, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

#Concatenate all DataFrames in the list into a single DataFrame
combined_data_dumper = pd.concat(dfs, ignore_index=True)

print("Fetched and Concatenated all files")
"""
print("Reading_csv")
combined_data_dumper = pd.read_csv('/Users/E2074/local-datasets/customer/appography/appography_data_last_sync/appography-data-may-june.csv')
print("Merging")

#Only selected city
df_combined_data_dumper = pd.merge(df_customer_demo_data['customer_id'],
                                   combined_data_dumper,
                                   how='inner',
                                   left_on=['customer_id'],
                                   right_on=['user_id'] 
                                  ) 
print("Merged")

#delete combined_data_dumper
del combined_data_dumper

#Latest updated data
df_combined_data_dumper['last_sync_date'] = df_combined_data_dumper['last_sync_date'].astype('int32')
# df_combined_data_dumper = df_combined_data_dumper.sort_values(by=['user_id', 'last_sync_date'], ascending=[True, False])
# df_combined_data_dumper = df_combined_data_dumper.drop_duplicates(subset='user_id', keep='first')
df_combined_data_dumper = df_combined_data_dumper[['customer_id', 'app_list_set']]

print("Updated data")

#regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_combined_data_dumper['app_list_set'] = df_combined_data_dumper['app_list_set'].apply(lambda x: extract_and_trim(x))
df_customer_appography_data = df_combined_data_dumper[['customer_id', 'app_list_set']]

print("regular-exp")

#delete df_combined_data_dumper
del df_combined_data_dumper

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time : ", execution_time, " mins")

In [ ]:
start_time = time.time()
# df_customer_appography_data.to_csv(local_database + 'raw_customer_appography_data_june.csv', index=False)
df_customer_appography_data = pd.read_csv(local_database + 'raw_customer_appography_data_june.csv')

# regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customer_appography_data['app_list_set'] = df_customer_appography_data['app_list_set'].apply(lambda x: extract_and_trim(x))
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time : ", execution_time, " mins")

### Merge

In [ ]:
df_active_customer_base.head(1)

In [ ]:
df_customer_segments.head(1)

In [ ]:
df_customer_demo_data.head(1)

In [ ]:
df_customer_use_case_data.head(1)

In [ ]:
df_customer_appography_data.head(1)

In [ ]:
start_time = time.time()
print('Merging 1.. ')
merge1 = pd.merge(df_active_customer_base,df_customer_segments,how='left',on = ['customer_id'])

print('Merging 2..')
merge2 = pd.merge(merge1,df_customer_demo_data,how='left',on = ['customer_id'])

print('Merging 3..')
merge3 = pd.merge(merge2,df_customer_use_case_data,how='left',on = ['customer_id'])

print('Merging 4..')
merge4 = pd.merge(merge3,df_customer_appography_data,how='left',on = ['customer_id'])

df_all_customer = merge4

del merge1
del merge2
del merge3
del merge4

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_all_customer.head(1)

In [ ]:
# df_all_customer

In [ ]:
df_all_customer

In [6]:
# df_all_customer.to_csv(local_database + 'processed_all_customer_june.csv', index=False)
df_all_customer = pd.read_csv(local_database + 'processed_all_customer_june.csv')

# regular-exp 
def extract_and_trim(text):
    if isinstance(text, str):
        return [val.strip() for val in re.findall(r"'([^']*)'", text)]
    else:
        return []

df_all_customer['app_list_set'] = df_all_customer['app_list_set'].apply(lambda x: extract_and_trim(str(x)))

In [7]:
df_all_customer.head(1)

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,android,61fa7cabed2eba2a567f64ad,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,16,MALE,NaN,NaN,leisure,NaN,"[google sheets, instagram, youtube, spotify, g..."


In [8]:
df_all_customer[df_all_customer['customer_id'] == '636e6d230620f059358ad78d']

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
58,android,636e6d230620f059358ad78d,26,6,2,13,13,4,2,PHH,17.0,HOOK,ONLY_AUTO,0,MALE,Unknown,NaN,Unknown,NaN,[]


In [11]:
# Replace empty lists with NaN
df_all_customer['app_list_set'] = df_all_customer['app_list_set'].apply(lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x)
df_all_customer[df_all_customer['customer_id'] == '636e6d230620f059358ad78d']

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
58,android,636e6d230620f059358ad78d,26,6,2,13,13,4,2,PHH,17.0,HOOK,ONLY_AUTO,0,MALE,Unknown,NaN,Unknown,NaN,NaN


In [12]:
df_all_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3245574 entries, 0 to 3245573
Data columns (total 20 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   platform                      object 
 1   customer_id                   object 
 2   fe_count                      int32  
 3   rr_count                      int32  
 4   net_count                     int32  
 5   morning_fe                    int32  
 6   evening_fe                    int32  
 7   morning_rr                    int32  
 8   evening_rr                    int32  
 9   ltr_segment                   object 
 10  life_time_rides               float64
 11  life_time_stage               object 
 12  service_affinity              object 
 13  age                           int32  
 14  gender                        object 
 15  first_half_drop_primary       object 
 16  first_half_drop_secondary     object 
 17  second_half_pickup_primary    object 
 18  second_half_pickup_sec

In [14]:
df_all_customer['fe_count'] = df_all_customer['fe_count'].astype('int32')
df_all_customer['rr_count'] = df_all_customer['rr_count'].astype('int32')
df_all_customer['net_count'] = df_all_customer['net_count'].astype('int32')
df_all_customer['morning_fe'] = df_all_customer['morning_fe'].astype('int32')
df_all_customer['evening_fe'] = df_all_customer['evening_fe'].astype('int32')
df_all_customer['morning_rr'] = df_all_customer['morning_rr'].astype('int32')
df_all_customer['evening_rr'] = df_all_customer['evening_rr'].astype('int32')
df_all_customer['age'] = df_all_customer['age'].astype('int32')

df_all_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3245574 entries, 0 to 3245573
Data columns (total 20 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   platform                      object 
 1   customer_id                   object 
 2   fe_count                      int32  
 3   rr_count                      int32  
 4   net_count                     int32  
 5   morning_fe                    int32  
 6   evening_fe                    int32  
 7   morning_rr                    int32  
 8   evening_rr                    int32  
 9   ltr_segment                   object 
 10  life_time_rides               float64
 11  life_time_stage               object 
 12  service_affinity              object 
 13  age                           int32  
 14  gender                        object 
 15  first_half_drop_primary       object 
 16  first_half_drop_secondary     object 
 17  second_half_pickup_primary    object 
 18  second_half_pickup_sec

## Distribution check

### Platform customer distribution

In [15]:
## platform customer distribution
df_temp = df_all_customer.groupby(['platform']).agg(customers = ('customer_id', 'nunique')).reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,customers,% distribution
0,android,2644254,81.47
1,ios,601320,18.53


### Age customer distribution

In [16]:
df_all_customer['age_presence'] = np.where(df_all_customer['age'] > 0, 'AVAILABLE', 'NA')

df_temp = df_all_customer\
                .groupby(['age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,age_presence,customers,% distribution
0,AVAILABLE,1071482,33.09
1,NA,2166969,66.91


In [17]:
df_temp = df_all_customer\
                .groupby(['platform', 'age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,age_presence,customers,total,% distribution
0,android,AVAILABLE,840053,2644254,31.77
1,android,NA,1804201,2644254,68.23
2,ios,AVAILABLE,234478,601320,38.99
3,ios,NA,366842,601320,61.01


### Gender distribution

In [18]:
df_temp = df_all_customer\
                .groupby(['gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,gender,customers,% distribution
0,FEMALE,912911,28.19
1,MALE,2238797,69.13
2,OTHERS,10709,0.33
3,UNKNOWN,76034,2.35


In [19]:
df_temp = df_all_customer\
                .groupby(['platform', 'gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,gender,customers,total,% distribution
0,android,FEMALE,712641,2644254,26.95
1,android,MALE,1847393,2644254,69.86
2,android,OTHERS,9516,2644254,0.36
3,android,UNKNOWN,74704,2644254,2.83
4,ios,FEMALE,202370,601320,33.65
5,ios,MALE,396411,601320,65.92
6,ios,OTHERS,1209,601320,0.20
7,ios,UNKNOWN,1330,601320,0.22


### Appography distribution

In [20]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_list_set'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,appo_presence,customers,% distribution
0,android,AVAILABLE,2576869,97.45
1,android,NA,67385,2.55


### Opportunity Size

In [21]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_list_set'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'age_presence', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,age_presence,appo_presence,customers,% distribution
0,android,AVAILABLE,AVAILABLE,820186,31.02
1,android,AVAILABLE,NA,19867,0.75
2,android,NA,AVAILABLE,1756683,66.43
3,android,NA,NA,47518,1.80


In [22]:
df_temp.pivot(index ='age_presence' , columns ='appo_presence', values =['% distribution'])

% distribution      
appo_presence      AVAILABLE    NA
age_presence                      
AVAILABLE              31.02  0.75
NA                     66.43  1.80

## One-Hot-Encoder

In [23]:
categorical_columns = ['gender', 'ltr_segment', 'life_time_stage', 'service_affinity', 
                       'first_half_drop_primary', 'first_half_drop_secondary',
                       'second_half_pickup_primary', 'second_half_pickup_secondary']

def one_hot_encoder(df):
    df_categorical = df[categorical_columns].copy()
    
    for column_name in categorical_columns:
        df_categorical[column_name] = df_categorical[column_name].fillna(column_name + '_unknown')
    
    # Initialize the OneHotEncoder
    encoder = OneHotEncoder()
    
    # Fit and transform the categorical columns
    encoded_categorical = encoder.fit_transform(df_categorical)
    
    # Convert the encoded result into a DataFrame
    encoded_df = pd.DataFrame(encoded_categorical.toarray(), 
                              columns=encoder.get_feature_names_out(input_features=categorical_columns))
    
    # Now drop the original categorical columns from the original DataFrame
    df = df.drop(columns=categorical_columns)

    # Concatenate the original DataFrame with the encoded DataFrame
    df = pd.concat([df, encoded_df], axis=1)
    
    
    return df

In [24]:
df_one_hot_encoded = one_hot_encoder(df_all_customer)
print(df_one_hot_encoded.shape)
df_one_hot_encoded['life_time_rides'].fillna(0, inplace=True)
df_one_hot_encoded.head(3)

(3245574, 89)


,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,life_time_rides,age,app_list_set,age_presence,appo_presence,gender_FEMALE,gender_MALE,gender_OTHERS,gender_UNKNOWN,ltr_segment_HH,ltr_segment_PHH,ltr_segment_UNKNOWN,ltr_segment_ltr_segment_unknown,life_time_stage_CHURN_OTB,life_time_stage_COMMITTED,life_time_stage_DETOX,...,first_half_drop_secondary_transit_station,second_half_pickup_primary_Unknown,second_half_pickup_primary_educational,second_half_pickup_primary_food,second_half_pickup_primary_govt_amenity,second_half_pickup_primary_health_and_personal,second_half_pickup_primary_household_needs,second_half_pickup_primary_leisure,second_half_pickup_primary_office,second_half_pickup_primary_place_of_worship,second_half_pickup_primary_residential,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_primary_transit_station,second_half_pickup_secondary_Unknown,second_half_pickup_secondary_educational,second_half_pickup_secondary_food,second_half_pickup_secondary_govt_amenity,second_half_pickup_secondary_health_and_personal,second_half_pickup_secondary_household_needs,second_half_pickup_secondary_leisure,second_half_pickup_secondary_office,second_half_pickup_secondary_place_of_worship,second_half_pickup_secondary_residential,second_half_pickup_secondary_second_half_pickup_secondary_unknown,second_half_pickup_secondary_transit_station
0,android,61fa7cabed2eba2a567f64ad,6,5,2,0,2,0,2,106.0,16,"[google sheets, instagram, youtube, spotify, g...",AVAILABLE,AVAILABLE,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,android,5ba4b9644ddb7f3de0913319,14,4,3,4,4,1,2,132.0,34,"[practo, angel broking, bigbasket, magicbricks...",AVAILABLE,AVAILABLE,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,android,64d706f0de2d9fadc9f70069,2,0,0,0,2,0,0,30.0,0,"[disney+ hotstar, flipkart, zomato, google map...",NA,AVAILABLE,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [25]:
df_one_hot_encoded.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr',
       'life_time_rides', 'age', 'app_list_set', 'age_presence',
       'appo_presence', 'gender_FEMALE', 'gender_MALE', 'gender_OTHERS',
       'gender_UNKNOWN', 'ltr_segment_HH', 'ltr_segment_PHH',
       'ltr_segment_UNKNOWN', 'ltr_segment_ltr_segment_unknown',
       'life_time_stage_CHURN_OTB', 'life_time_stage_COMMITTED',
       'life_time_stage_DETOX', 'life_time_stage_DORMANT',
       'life_time_stage_HANDHOLDING', 'life_time_stage_HOOK',
       'life_time_stage_SOFT_CHURN', 'life_time_stage_SUSTENANCE',
       'life_time_stage_UNKNOWN', 'life_time_stage_life_time_stage_unknown',
       'service_affinity_AUTO_CAB', 'service_affinity_LINK_AUTO',
       'service_affinity_LINK_CAB', 'service_affinity_NO_AFFINITY',
       'service_affinity_ONLY_AUTO', 'service_affinity_ONLY_CAB',
       'service_affinity_ONLY_LINK', 'service_affinity_UNKNOWN',
      

In [26]:
encoded_columns = ['customer_id', 
#                    'fe_count', 'rr_count', 'net_count',
#                    'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', # 'life_time_rides', 
#                    'gender_FEMALE', 'gender_MALE', 'gender_OTHERS','gender_UNKNOWN', 
                   'ltr_segment_HH', 'ltr_segment_PHH', 'ltr_segment_UNKNOWN', 
                   'life_time_stage_CHURN_OTB', 'life_time_stage_COMMITTED', 'life_time_stage_DETOX', 
                   'life_time_stage_DORMANT', 'life_time_stage_HANDHOLDING', 'life_time_stage_HOOK',
                   'life_time_stage_SOFT_CHURN', 'life_time_stage_SUSTENANCE', 'life_time_stage_UNKNOWN', 
                   'service_affinity_AUTO_CAB', 'service_affinity_LINK_AUTO','service_affinity_LINK_CAB', 
                   'service_affinity_NO_AFFINITY', 'service_affinity_ONLY_AUTO', 'service_affinity_ONLY_CAB',
                   'service_affinity_ONLY_LINK', 'service_affinity_UNKNOWN',
                   'first_half_drop_primary_Unknown', 
                   'first_half_drop_primary_educational',
                   'first_half_drop_primary_first_half_drop_primary_unknown',
                   'first_half_drop_primary_food', 'first_half_drop_primary_govt_amenity',
                   'first_half_drop_primary_health_and_personal',
                   'first_half_drop_primary_household_needs',
                   'first_half_drop_primary_leisure', 'first_half_drop_primary_office',
                   'first_half_drop_primary_place_of_worship',
                   'first_half_drop_primary_residential',
                   'first_half_drop_primary_transit_station',
                   'first_half_drop_secondary_Unknown',
                   'first_half_drop_secondary_educational',
                   'first_half_drop_secondary_first_half_drop_secondary_unknown',
                   'first_half_drop_secondary_food',
                   'first_half_drop_secondary_govt_amenity',
                   'first_half_drop_secondary_health_and_personal',
                   'first_half_drop_secondary_household_needs',
                   'first_half_drop_secondary_leisure', 'first_half_drop_secondary_office',
                   'first_half_drop_secondary_place_of_worship',
                   'first_half_drop_secondary_residential',
                   'first_half_drop_secondary_transit_station',
                   'second_half_pickup_primary_Unknown',
                   'second_half_pickup_primary_educational',
                   'second_half_pickup_primary_food',
                   'second_half_pickup_primary_govt_amenity',
                   'second_half_pickup_primary_health_and_personal',
                   'second_half_pickup_primary_household_needs',
                   'second_half_pickup_primary_leisure',
                   'second_half_pickup_primary_office',
                   'second_half_pickup_primary_place_of_worship',
                   'second_half_pickup_primary_residential',
                   'second_half_pickup_primary_second_half_pickup_primary_unknown',
                   'second_half_pickup_primary_transit_station',
                   'second_half_pickup_secondary_Unknown',
                   'second_half_pickup_secondary_educational',
                   'second_half_pickup_secondary_food',
                   'second_half_pickup_secondary_govt_amenity',
                   'second_half_pickup_secondary_health_and_personal',
                   'second_half_pickup_secondary_household_needs',
                   'second_half_pickup_secondary_leisure',
                   'second_half_pickup_secondary_office',
                   'second_half_pickup_secondary_place_of_worship',
                   'second_half_pickup_secondary_residential',
                   'second_half_pickup_secondary_second_half_pickup_secondary_unknown',
                   'second_half_pickup_secondary_transit_station']

df_one_hot_encoded = df_one_hot_encoded[encoded_columns]

# Appography Age Bins Classification

In [27]:
df_all_customer.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 'ltr_segment',
       'life_time_rides', 'life_time_stage', 'service_affinity', 'age',
       'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary',
       'app_list_set', 'age_presence', 'appo_presence'],
      dtype='object')

In [28]:
df_triangulation_1 = df_all_customer[(df_all_customer['platform'].isin(['android']) 
                                      & 
                                      df_all_customer['appo_presence'].isin(['AVAILABLE']))]\
                                    [['customer_id', 'age', 'appo_presence','app_list_set',
                                      'fe_count', 'rr_count', 'net_count',
                                      'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 
                                      'ltr_segment', 'life_time_rides', 'life_time_stage', 'service_affinity',
                                      'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
                                      'second_half_pickup_primary', 'second_half_pickup_secondary',
                                     ]]

df_triangulation_1.head(5)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,"[google sheets, instagram, youtube, spotify, g...",6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN
1,5ba4b9644ddb7f3de0913319,34,AVAILABLE,"[practo, angel broking, bigbasket, magicbricks...",14,4,3,4,4,1,2,PHH,132.0,SUSTENANCE,ONLY_LINK,MALE,residential,NaN,residential,household_needs
2,64d706f0de2d9fadc9f70069,0,AVAILABLE,"[disney+ hotstar, flipkart, zomato, google map...",2,0,0,0,2,0,0,PHH,30.0,DORMANT,ONLY_AUTO,FEMALE,NaN,NaN,NaN,NaN
4,5f8ef77b8f17c566c519574d,35,AVAILABLE,"[zomato, facebook, messenger, google news, goo...",46,11,5,11,32,4,6,PHH,55.0,HOOK,ONLY_LINK,MALE,food,office,food,residential
6,632c059b868f89c8c8b13f7b,28,AVAILABLE,"[google calendar, linkedin, paytm money, truec...",22,9,1,2,19,0,9,PHH,5.0,DORMANT,ONLY_LINK,MALE,NaN,NaN,office,NaN


### Explode

In [29]:
## Explode list into individual rows

df_appography_explode = df_triangulation_1.explode('app_list_set')
df_appography_explode['app_list_set'] = df_appography_explode['app_list_set'].str.lower()

df_appography_explode.head(4)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,google sheets,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,instagram,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,youtube,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,spotify,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN


### Mobile app to category mapping

In [30]:
category_list = pd.read_csv('/Users/E2074/local-datasets/customer/appography/category_mapping.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list['category'] = category_list['category'].str.lower()
category_list = category_list[['app_name', 'category']]
print(category_list.shape)
print(category_list.category.unique())
category_list.head(1)

(538, 2)
['educational' 'ott' 'delivery_grocery' 'tools' 'office' 'dating' 'news'
 'devotional' 'travel_metro' 'gaming' 'travel_bookings'
 'finance_transactions' 'ecommerce' 'streaming_music' 'finance_investment'
 'health' 'fantasy_sports' 'delivery_food' 'home_security' 'finance_news'
 'travel_ridehailing' 'social' 'telecom' 'entertainment' 'news_finance'
 'wearable' 'driver_app' 'home_service' 'insurance' 'job_search' 'vehicle'
 'government']


,app_name,category
0,ignou e-content,educational


In [33]:
df_main_apps_explode = df_appography_explode #.copy()

### Merge

In [34]:
df_main_apps_explode.head(2)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,google sheets,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,instagram,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN


In [35]:
category_list.head(2)

,app_name,category,top_categories,last_categories
0,ignou e-content,educational,rest,educational
1,aha,ott,rest,ott


In [36]:
df_main_app_category =  pd.merge(df_main_apps_explode, 
                                 category_list[['app_name', 'category']], 
                                 how='left', 
                                 left_on='app_list_set', 
                                 right_on='app_name'
                              )
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,category
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,google sheets,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,google sheets,tools
1,61fa7cabed2eba2a567f64ad,16,AVAILABLE,instagram,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,instagram,social
2,61fa7cabed2eba2a567f64ad,16,AVAILABLE,youtube,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,youtube,tools
3,61fa7cabed2eba2a567f64ad,16,AVAILABLE,spotify,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,spotify,streaming_music


#### Distribution check on categories

In [37]:
total_customers = df_main_app_category.customer_id.nunique()
total_customers

df_temp = df_main_app_category\
            .groupby(['category'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,category,app_list,customers,%
0,tools,"[money lover, google home, weather & radar, li...",2566163,99.58
1,social,"[mx takatak, roposo, twitter, telegram, whatsa...",2551708,99.02
2,ott,"[zee5, disney+ hotstar, shemaroome livetv, net...",2128327,82.59
3,ecommerce,"[shopsy, tata cliq, pepperfry, paytm mall, voo...",2060129,79.95
4,finance_transactions,"[airtel thanks, paytm, paytm money, imobile pa...",2016177,78.24
5,travel_ridehailing,"[uts, yulu, namma yatri, in drive, quick ride,...",1846376,71.65
6,streaming_music,"[jiosaavn, soundcloud, spotify, wynk music, ap...",1715179,66.56
7,delivery_food,"[starbucks india, pizza hut india, uber eats, ...",1588338,61.64
8,office,"[microsoft 365, trello, ms authenticator, zoho...",1449571,56.25
9,news,"[abp live, hindustan times, ndtv news, the ind...",1077243,41.80


#### Most Common used apps

In [38]:
df_temp = df_main_app_category\
            .groupby(['app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,app_list_set,customers,%
0,google maps,2545130,98.77
1,youtube,2539282,98.54
2,gmail,2539036,98.53
3,whatsapp,2484896,96.43
4,google photos,2411749,93.59
...,...,...,...
419,likee,2,0.00
420,afterlight,1,0.00
421,shemaroome,1,0.00
422,bansal classes,1,0.00


In [39]:
most_common_apps = ['google maps', 'gmail', 'youtube', 'whatsapp', 'clock', 'calculator',
                    'google calendar', 'google photos']

### Way 1 (All Apps Level)

#### Transform Categories 

In [40]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,category
0,61fa7cabed2eba2a567f64ad,16,AVAILABLE,google sheets,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,google sheets,tools
1,61fa7cabed2eba2a567f64ad,16,AVAILABLE,instagram,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,instagram,social
2,61fa7cabed2eba2a567f64ad,16,AVAILABLE,youtube,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,youtube,tools
3,61fa7cabed2eba2a567f64ad,16,AVAILABLE,spotify,6,5,2,0,2,0,2,PHH,106.0,SOFT_CHURN,ONLY_AUTO,MALE,NaN,NaN,leisure,NaN,spotify,streaming_music


In [41]:
# df_main_app_category[df_main_app_category['app_list_set'].isna()].nunique()

In [42]:
df_customer_mapped_v1 = df_main_app_category[~df_main_app_category['app_list_set'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             category = ('category', lambda x: list(set(x)))
                            )\
                        .reset_index()

df_customer_mapped_v1.head(4)

,customer_id,age,app_list,category
0,5737c6adddbec2203f73316d,0,"[paytm money, disney+ hotstar, telegram, wynk ...","[streaming_music, driver_app, ott, tools, soci..."
1,5737c6aeddbec2203f733176,35,"[airtel thanks, ms edge, microsoft 365, micros...","[ott, office, delivery_food, travel_ridehailin..."
2,5737c6b1ddbec2203f73318b,33,"[ixigo, nobroker, twitter, borzo driver app, t...","[driver_app, ott, home_service, delivery_food,..."
3,5737c6baddbec2203f7331d9,0,"[nobroker, spotify, bookmyshow, telegram, uber...","[streaming_music, ott, home_service, delivery_..."


In [43]:
unique_categories = set(category for sublist in df_customer_mapped_v1['app_list'] for category in sublist)
unique_categories_list = list(unique_categories)
print("Count of unique apps:", len(unique_categories))
# sorted(unique_categories)

Count of unique apps: 416


In [44]:
df_customer_mapped_v1.head(2)

,customer_id,age,app_list,category
0,5737c6adddbec2203f73316d,0,"[paytm money, disney+ hotstar, telegram, wynk ...","[streaming_music, driver_app, ott, tools, soci..."
1,5737c6aeddbec2203f733176,35,"[airtel thanks, ms edge, microsoft 365, micros...","[ott, office, delivery_food, travel_ridehailin..."


In [45]:
df_customer_mapped_v1.shape

(2574658, 4)

In [46]:
for category in unique_categories:
    df_customer_mapped_v1[category] = df_customer_mapped_v1['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v1.head(1)

,customer_id,age,app_list,category,spencers,the indian express,et money,eros now,zepto driver app,icicidirect: stocks,groupme,signal,nestaway,disney+ hotstar,amazon shopping,amazon music,mycbseguide,ms edge,espncricinfo,zivame,sun nxt,bloomberg,bookmyshow,ms authenticator,spicejet,...,zoom,economic times,home workout - no equipment,zoho meeting,amazon prime video,officesuite,linkedin,aha,apple music,google news,ncert books,abhibus,hdfc mobile banking,limeroad,kite by zerodha,meesho,google pay,"s pizza"",",ola driver app,brainly,zee5,nykaa man,zerodha kite,mobikwik,ludo king
0,5737c6adddbec2203f73316d,0,"[paytm money, disney+ hotstar, telegram, wynk ...","[streaming_music, driver_app, ott, tools, soci...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [47]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['Below-21', '21-25',  '26-30', '31-35', '36-45', 'Above-45]

df_customer_mapped_v1['age_group'] = pd.cut(df_customer_mapped_v1['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [48]:
main_list = ['customer_id', 'age', 'age_group' ,'category']
unique_column_list = main_list + unique_categories_list

In [49]:
df_customer_mapped_v1 = df_customer_mapped_v1[unique_column_list]
df_customer_mapped_v1.head(1)

,customer_id,age,age_group,category,spencers,the indian express,et money,eros now,zepto driver app,icicidirect: stocks,groupme,signal,nestaway,disney+ hotstar,amazon shopping,amazon music,mycbseguide,ms edge,espncricinfo,zivame,sun nxt,bloomberg,bookmyshow,ms authenticator,spicejet,...,zoom,economic times,home workout - no equipment,zoho meeting,amazon prime video,officesuite,linkedin,aha,apple music,google news,ncert books,abhibus,hdfc mobile banking,limeroad,kite by zerodha,meesho,google pay,"s pizza"",",ola driver app,brainly,zee5,nykaa man,zerodha kite,mobikwik,ludo king
0,5737c6adddbec2203f73316d,0,NaN,"[streaming_music, driver_app, ott, tools, soci...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


#### OneHotEncoder

In [50]:
# Adding required additional info by one-hot-encoding 

df_customer_mapped_v1 = pd.merge(df_customer_mapped_v1, 
                                 df_one_hot_encoded,
                                 how='left',
                                 left_on='customer_id',
                                 right_on='customer_id'
                                )
# df_customer_mapped_v1['life_time_rides'].fillna(0, inplace=True)

df_customer_mapped_v1.head(1)

,customer_id,age,age_group,category,spencers,the indian express,et money,eros now,zepto driver app,icicidirect: stocks,groupme,signal,nestaway,disney+ hotstar,amazon shopping,amazon music,mycbseguide,ms edge,espncricinfo,zivame,sun nxt,bloomberg,bookmyshow,ms authenticator,spicejet,...,first_half_drop_secondary_transit_station,second_half_pickup_primary_Unknown,second_half_pickup_primary_educational,second_half_pickup_primary_food,second_half_pickup_primary_govt_amenity,second_half_pickup_primary_health_and_personal,second_half_pickup_primary_household_needs,second_half_pickup_primary_leisure,second_half_pickup_primary_office,second_half_pickup_primary_place_of_worship,second_half_pickup_primary_residential,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_primary_transit_station,second_half_pickup_secondary_Unknown,second_half_pickup_secondary_educational,second_half_pickup_secondary_food,second_half_pickup_secondary_govt_amenity,second_half_pickup_secondary_health_and_personal,second_half_pickup_secondary_household_needs,second_half_pickup_secondary_leisure,second_half_pickup_secondary_office,second_half_pickup_secondary_place_of_worship,second_half_pickup_secondary_residential,second_half_pickup_secondary_second_half_pickup_secondary_unknown,second_half_pickup_secondary_transit_station
0,5737c6adddbec2203f73316d,0,NaN,"[streaming_music, driver_app, ott, tools, soci...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [51]:
# Find columns with NaN values
columns_with_nan = df_customer_mapped_v1.columns[df_customer_mapped_v1.isna().any()].tolist()

# Filter DataFrame to get only columns with NaN values
df_with_nan = df_customer_mapped_v1[columns_with_nan]
df_with_nan.isnull().sum()

age_group    1758797
dtype: int64

In [52]:
# df_customer_mapped_v1['life_time_rides'].fillna(0, inplace=True)

### Stratified Sampling & Model

In [62]:
df_customer_mapped_v1_age_w = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()]
df_customer_mapped_v1_age_na = df_customer_mapped_v1[df_customer_mapped_v1['age_group'].isna()]

# Keep a copy of the columns to be retained
df_customer_mapped_v1_age_na_columns = df_customer_mapped_v1_age_na[['customer_id', 'category', 'age']].copy()

# Drop unnecessary columns for model training
df_customer_mapped_v1_age_w.drop(columns=['customer_id', 'category', 'age'], axis=1, inplace=True)
df_customer_mapped_v1_age_na.drop(columns=['customer_id', 'category', 'age'], axis=1, inplace=True)

In [63]:
def evaluate_target_vs_rest(df, target_column):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    classifiers = {}
    
    for target_class in target_classes:
        
        # Create a copy of df to avoid modifying the original dataframe
        df_temp = df.copy()
        
        # Split the bins target_class vs Rest 
        df_target = df_temp[df_temp[target_column] == target_class]
        df_rest = df_temp[df_temp[target_column] != target_class]
        
        df_other_sample = df_rest.sample(n=df_target.shape[0], random_state=1)
        df_combined = pd.concat([df_target, df_other_sample], ignore_index=True)
        
        # Create a binary target column where target_class is 1 and rest are 0
        df_combined['target'] = df_combined[target_column].apply(lambda x: 1 if x == target_class else 0)

        # Ensure all column names are strings
        df_combined.columns = df_combined.columns.astype(str)

        # train_test_split
        X_train, X_test, y_train, y_test = train_test_split(df_combined.drop(columns=[target_column, 'target']), 
                                                            df_combined['target'], 
                                                            test_size=0.2, 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        classifiers[target_class] = rf_model

        # Extract feature importances
        feature_importances = rf_model.feature_importances_

        # Get the indices of top features
        top_feature_indices = np.argsort(feature_importances)[::-1][:20] # Adjust 20 as needed
        # Get the names of top features
        top_features = X_train.columns[top_feature_indices]

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest:")
        print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        print("Precision : ", round(precision * 100.00, 2), "%")
        print("Recall : ", round(recall * 100.00, 2), "%")
        print("F1 Score : ", round(f1 * 100.00, 2), "%")
        print("Top Features:")
        for feature in top_features:
            print(feature)
        print("---------------------------------------")
        
    return classifiers

In [64]:
classifiers = evaluate_target_vs_rest(df_customer_mapped_v1_age_w, 'age_group')

Results for class 15-20 vs Rest:
Accuracy :  71.7 %
Precision :  73.05 %
Recall :  68.97 %
F1 Score :  70.95 %
Top Features:
cred
free fire max
snapchat
paytm money
hdfc mobile banking
messenger
linkedin
axis mobile
spotify
telegram
airtel thanks
jio cinema
ms authenticator
truecaller
uber
disney+ hotstar
ola
flipkart
microsoft teams
amazon prime video
---------------------------------------
Results for class 21-25 vs Rest:
Accuracy :  71.27 %
Precision :  71.29 %
Recall :  71.45 %
F1 Score :  71.37 %
Top Features:
snapchat
messenger
google classroom
telegram
cred
spotify
instagram
airtel thanks
facebook
jio cinema
urban company
disney+ hotstar
paytm money
truecaller
google news
uber
wynk music
netflix
second_half_pickup_primary_second_half_pickup_primary_unknown
service_affinity_ONLY_LINK
---------------------------------------
Results for class 26-30 vs Rest:
Accuracy :  61.43 %
Precision :  61.4 %
Recall :  61.32 %
F1 Score :  61.36 %
Top Features:
airtel thanks
snapchat
truecaller


In [65]:
classifiers

{'15-20': RandomForestClassifier(random_state=42),
 '21-25': RandomForestClassifier(random_state=42),
 '26-30': RandomForestClassifier(random_state=42),
 '31-35': RandomForestClassifier(random_state=42),
 '36-45': RandomForestClassifier(random_state=42),
 '46-60': RandomForestClassifier(random_state=42)}

In [100]:
def predict_age_group(df, classifiers):
    predictions = pd.DataFrame(index=df.index)
    
    for target_class, classifier in classifiers.items():
        # Ensure the features used in prediction match the training data
        X = df[[col for col in df.columns if col != 'age_group']]
        predictions[f'{target_class}'] = classifier.predict_proba(X)[:, 1]
        
    binary_predictions = (predictions > 0.5).astype(int)
    
    return binary_predictions

In [101]:
# Use the models to predict the age group for df_customer_mapped_v1_age_na
predictions = predict_age_group(df_customer_mapped_v1_age_na, classifiers)

In [102]:
predictions

,15-20,21-25,26-30,31-35,36-45,46-60
0,0,1,1,0,0,0
3,0,0,0,1,1,0
4,1,0,0,0,1,0
5,1,0,1,0,0,0
6,0,0,0,1,1,1
...,...,...,...,...,...,...
2581582,0,1,1,0,0,0
2581583,0,1,0,1,1,0
2581584,1,1,0,0,0,0
2581585,0,0,0,1,1,1


In [103]:
predictions['total_sum'] = predictions.sum(axis=1)

In [125]:
# Merge the original columns with the prediction results
df_customer_mapped_v1_age_na_with_predictions = pd.concat([df_customer_mapped_v1_age_na_columns['customer_id'], predictions], axis=1)

In [126]:
df_customer_mapped_v1_age_na_with_predictions

,customer_id,15-20,21-25,26-30,31-35,36-45,46-60,total_sum
0,5737c6adddbec2203f73316d,0,1,1,0,0,0,2
3,5737c6baddbec2203f7331d9,0,0,0,1,1,0,2
4,5737c6bbddbec2203f7331e5,1,0,0,0,1,0,2
5,5737c6bbddbec2203f7331e8,1,0,1,0,0,0,2
6,5737c6c0ddbec2203f733215,0,0,0,1,1,1,3
...,...,...,...,...,...,...,...,...
2581582,6681a37497221018034beedc,0,1,1,0,0,0,2
2581583,6681a37f684ea609676ba69e,0,1,0,1,1,0,3
2581584,6681a38a8641fbb5b5d22afd,1,1,0,0,0,0,2
2581585,6681a3a5684ea648986bac3a,0,0,0,1,1,1,3


In [127]:
df_customer_mapped_v1_age_na_with_predictions.total_sum.value_counts(normalize=True)

total_sum
2    0.467516
3    0.328257
1    0.104436
4    0.085121
0    0.008036
5    0.006437
6    0.000197
Name: proportion, dtype: float64

In [166]:
# Define a function to apply the logic
def age_weight_function(row):
    cols_with_one = [col for col in ['15-20', '21-25', '26-30', '31-35', '36-45', '46-60'] if row[col] == 1]
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return cols_with_one[0]
    elif total_sum == 2:
        return cols_with_one[-1]
    elif total_sum == 3 or total_sum == 5:
        return cols_with_one[len(cols_with_one) // 2]
    elif total_sum == 4:
        return cols_with_one[2]
    elif total_sum == 6:
        return cols_with_one[4]
    else:
        return 'UNKNOWN'

def get_confidence_tag(row):
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return 'VERY_HIGH'
    elif total_sum == 2:
        return 'HIGH'
    elif total_sum == 3:
        return 'MEDIUM'
    elif total_sum in [4, 5, 6]:
        return 'LOW'
    else:
        return 'UNKNOWN'

In [173]:
df_customer_mapped_v1_age_na_with_predictions['age_group'] = df_customer_mapped_v1_age_na_with_predictions.apply(age_weight_function, axis=1)
df_customer_mapped_v1_age_na_with_predictions['confidence_tag'] = df_customer_mapped_v1_age_na_with_predictions.apply(get_confidence_tag, axis=1)

In [174]:
df_customer_mapped_v1_age_na_with_predictions.predicted_age.value_counts(normalize=True)

predicted_age
21-25      0.333332
36-45      0.284944
26-30      0.160886
31-35      0.138192
46-60      0.060587
15-20      0.014022
unknown    0.008036
Name: proportion, dtype: float64

In [175]:
df_customer_mapped_v1_age_na_with_predictions.confidence_tag.value_counts(normalize=True)

confidence_tag
HIGH         0.467516
MEDIUM       0.328257
VERY_HIGH    0.104436
LOW          0.091755
UNKNOWN      0.008036
Name: proportion, dtype: float64

In [176]:
df_final2 = df_customer_mapped_v1_age_na_with_predictions[['customer_id', 'age_group', 'confidence_tag']]

In [177]:
df_final2

,customer_id,age_group,confidence_tag
0,5737c6adddbec2203f73316d,26-30,HIGH
3,5737c6baddbec2203f7331d9,36-45,HIGH
4,5737c6bbddbec2203f7331e5,36-45,HIGH
5,5737c6bbddbec2203f7331e8,26-30,HIGH
6,5737c6c0ddbec2203f733215,36-45,MEDIUM
...,...,...,...
2581582,6681a37497221018034beedc,26-30,HIGH
2581583,6681a37f684ea609676ba69e,31-35,MEDIUM
2581584,6681a38a8641fbb5b5d22afd,21-25,HIGH
2581585,6681a3a5684ea648986bac3a,36-45,MEDIUM


In [178]:
df_final1 = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()][['customer_id', 'age_group']]

In [179]:
df_final1['confidence_tag'] = 'VERY_HIGH'
df_final1

,customer_id,age_group,confidence_tag
1,5737c6aeddbec2203f733176,36-45,VERY_HIGH
2,5737c6b1ddbec2203f73318b,31-35,VERY_HIGH
8,5737c6c5ddbec2203f733239,26-30,VERY_HIGH
10,5737c6deddbec2203f73331a,36-45,VERY_HIGH
12,5737c6e3ddbec2203f733341,26-30,VERY_HIGH
...,...,...,...
2581394,6681939a3d028a1002d41600,15-20,VERY_HIGH
2581474,668198c821fb388f022c59b8,31-35,VERY_HIGH
2581535,66819daa684ea639526ae36a,21-25,VERY_HIGH
2581560,6681a0b28989b120099bf178,15-20,VERY_HIGH


In [181]:
df_predicated_age_bins = pd.concat([df_final1,df_final2])

In [182]:
df_predicated_age_bins.shape

(2581587, 3)